# Seq2Seq Anh → Việt · GRU Encoder–Decoder + Attention

Bản dành cho **Kaggle GPU**, train từ đầu trên dữ liệu song ngữ của dự án.

1. Đặt `train.tsv`, `validation.tsv`, `test.tsv` trong `/kaggle/working/Encode_Decode_NLP/data`.
2. Bật **Settings → Accelerator → GPU**, import notebook này rồi **Run All**.
3. Model, checkpoint và kết quả nằm trong `/kaggle/working/Encode_Decode_NLP/outputs/s2s/<run>/`.

Đường dẫn được khai báo trực tiếp trong cell cấu hình. Sửa `CFG` tại một nơi: mặc định dùng toàn bộ câu hợp lệ, batch 64, embedding/hidden 256, tối đa **50 epoch** và early stopping. Notebook dùng **một GPU CUDA**; nếu chọn T4 ×2 thì phiên bản này dùng GPU đầu tiên.

**Dữ liệu hiện tại:** `validation.tsv` trùng `test.tsv`. Vì vậy notebook tạo validation mới từ 5% nhóm câu nguồn trong train, loại câu nguồn test khỏi train và giữ test riêng. `validation.tsv` gốc không được dùng để chọn model.

Luồng chạy: dữ liệu → vocabulary → batch theo độ dài → encoder/decoder → train → đánh giá → dịch câu và xem attention. `best_model.pt` dùng để dịch; `last_model.pt` dùng để tiếp tục train.


## 1. Thư viện và cấu hình

Kaggle đã có PyTorch, NumPy, Matplotlib và tqdm; cell dưới cài SacreBLEU để tính BLEU/chrF. Bật Internet nếu gói chưa có. Dùng PyTorch 2.3+.

Notebook yêu cầu CUDA và in tên GPU ngay đầu phiên. Muốn thử nhanh, sửa `max_train_samples=5000`, `max_validation_samples=500`, `max_test_samples=500`, `epochs=3`. Train đầy đủ thì để ba giới hạn mẫu là `None`.


In [ ]:
get_ipython().run_line_magic("pip", "install -q sacrebleu")


In [ ]:
import csv
import html
import json
import math
import random
import re
import time
import unicodedata
from collections import Counter
from datetime import datetime
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence
from torch.utils.data import Dataset, DataLoader, Sampler
from tqdm.auto import tqdm
from sacrebleu.metrics import BLEU, CHRF


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def write_json(path, payload):
    path.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False),
        encoding="utf-8",
    )


In [ ]:
DATA_DIR = Path("/kaggle/working/Encode_Decode_NLP/data")
OUTPUT_DIR = Path("/kaggle/working/Encode_Decode_NLP/outputs/s2s")
RESUME_PATH = None  # Path("/kaggle/working/.../last_model.pt") để tiếp tục train.

CFG = {
    "seed": 42,
    "max_train_samples": None,
    "max_validation_samples": None,
    "max_test_samples": None,
    "validation_fraction": 0.05,
    "max_src_tokens": 60,
    "max_tgt_tokens": 60,
    "src_vocab_size": 30_000,
    "tgt_vocab_size": 30_000,
    "min_freq": 2,
    "embedding_dim": 256,
    "hidden_dim": 256,
    "dropout": 0.2,
    "batch_size": 64,
    "epochs": 50,
    "learning_rate": 1e-3,
    "gradient_clip": 1.0,
    "patience": 5,
    "num_workers": 2,
    "use_amp": True,
}

assert torch.cuda.is_available(), "Bật GPU trong Kaggle Settings → Accelerator rồi chạy lại."
DEVICE = torch.device("cuda:0")
resume = None
if RESUME_PATH is not None:
    resume = torch.load(RESUME_PATH, map_location="cpu", weights_only=True)
    CFG = {**resume["config"], "epochs": CFG["epochs"], "num_workers": CFG["num_workers"]}

USE_AMP = CFG["use_amp"]
set_seed(CFG["seed"])
torch.set_float32_matmul_precision("high")
RUN_DIR = OUTPUT_DIR / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))
print("Device:", DEVICE, "| AMP:", USE_AMP)
print("Data:", DATA_DIR)
print("Output:", RUN_DIR)
print(json.dumps(CFG, ensure_ascii=False, indent=2))


## 2. Dữ liệu và phép chia train/validation/test

Mỗi dòng TSV: `câu Anh<TAB>câu Việt`, không có header. Tokenizer chuẩn hóa Unicode/HTML, chuyển chữ thường và tách dấu câu; token tiếng Việt thường là âm tiết.

Loại câu rỗng, cặp trùng và câu dài hơn giới hạn trong `CFG`. Với bộ dữ liệu này, validation được tạo từ train **theo nhóm câu nguồn** vì file validation gốc trùng test. Vocabulary chỉ học từ train sau khi chia. Các file dữ liệu gốc được giữ nguyên.


In [ ]:
TOKEN_PATTERN = re.compile(r"\w+(?:['’]\w+)*|[^\w\s]", re.UNICODE)

def normalize_text(text):
    text = unicodedata.normalize("NFKC", html.unescape(text)).lower()
    return " ".join(text.split())

def tokenize(text):
    return TOKEN_PATTERN.findall(normalize_text(text))

def tokens_to_text(tokens):
    text = " ".join(tokens)
    text = re.sub(r"\s+([,.;:!?%)\]])", r"\1", text)
    return re.sub(r"([(\[])\s+", r"\1", text)

def read_tsv(path):
    pairs, seen = [], set()
    stats = {"rows": 0, "duplicates": 0, "empty": 0, "too_long": 0}
    # QUOTE_NONE: dấu " trong câu là nội dung, không phải quote của TSV.
    with path.open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.reader(handle, delimiter="\t", quoting=csv.QUOTE_NONE)
        for line_number, row in enumerate(reader, start=1):
            stats["rows"] += 1
            if len(row) != 2:
                raise ValueError(f"{path.name}:{line_number}: cần đúng 2 cột.")
            src_tokens, tgt_tokens = tokenize(row[0]), tokenize(row[1])
            if not src_tokens or not tgt_tokens:
                stats["empty"] += 1
                continue
            if (len(src_tokens) > CFG["max_src_tokens"]
                    or len(tgt_tokens) > CFG["max_tgt_tokens"]):
                stats["too_long"] += 1
                continue
            pair = (" ".join(src_tokens), " ".join(tgt_tokens))
            if pair in seen:
                stats["duplicates"] += 1
                continue
            seen.add(pair)
            pairs.append(pair)
    if not pairs:
        raise ValueError(f"{path.name} không có cặp hợp lệ sau lọc.")
    return pairs, stats


In [ ]:
raw_train, train_stats = read_tsv(DATA_DIR / "train.tsv")
raw_test, test_stats = read_tsv(DATA_DIR / "test.tsv")

# Tạo validation từ train; không dùng validation.tsv đang trùng test.tsv.
test_sources = {src for src, _ in raw_test}
train_pool = [(src, tgt) for src, tgt in raw_train if src not in test_sources]
sources = sorted({src for src, _ in train_pool})
random.Random(CFG["seed"]).shuffle(sources)
validation_sources = set(sources[:max(1, round(len(sources) * CFG["validation_fraction"]))])


def sample_pairs(pairs, limit, seed):
    pairs = list(pairs)
    random.Random(seed).shuffle(pairs)
    return pairs[:limit]  # [:None] giữ toàn bộ.


train_pairs = sample_pairs(
    [(src, tgt) for src, tgt in train_pool if src not in validation_sources],
    CFG["max_train_samples"], CFG["seed"],
)
validation_pairs = sample_pairs(
    [(src, tgt) for src, tgt in train_pool if src in validation_sources],
    CFG["max_validation_samples"], CFG["seed"] + 1,
)
test_pairs = sample_pairs(raw_test, CFG["max_test_samples"], CFG["seed"] + 2)

split_summary = {
    "input_stats": {"train": train_stats, "test": test_stats},
    "validation_source": "5% source groups from train; original validation equals test",
    "removed_train_pairs_with_test_source": len(raw_train) - len(train_pool),
    "selected_pairs": {
        "train": len(train_pairs), "validation": len(validation_pairs), "test": len(test_pairs),
    },
}
print("Train TSV:", train_stats)
print("Test TSV:", test_stats)
print("Số cặp dùng:", split_summary["selected_pairs"])
print("EN:", train_pairs[0][0])
print("VI:", train_pairs[0][1])


## 3. Vocabulary và batch theo độ dài

`PAD=0`, `UNK=1`, `BOS=2`, `EOS=3`. Source: câu Anh + EOS. Target: BOS + câu Việt + EOS.

Mã hóa dataset một lần. Train sampler gom các câu có độ dài gần nhau để giảm PAD, rồi xáo trộn thứ tự batch mỗi epoch. DataLoader dùng 2 worker và pinned memory. Lengths ở CPU phục vụ packed GRU; source/target và model ở GPU.


In [ ]:
SPECIAL_TOKENS = ["<pad>", "<unk>", "<bos>", "<eos>"]
PAD, UNK, BOS, EOS = range(4)

class Vocabulary:
    def __init__(self, itos):
        self.itos = list(itos)
        self.stoi = {token: idx for idx, token in enumerate(self.itos)}

    @classmethod
    def build(cls, token_sequences, max_size, min_freq):
        counts = Counter(token for sequence in token_sequences for token in sequence)
        # Tie-break theo chữ để mapping ổn định.
        ranked = sorted(counts.items(), key=lambda item: (-item[1], item[0]))
        tokens = [
            token for token, count in ranked
            if count >= min_freq and token not in SPECIAL_TOKENS
        ][:max_size - len(SPECIAL_TOKENS)]
        return cls(SPECIAL_TOKENS + tokens)

    def __len__(self):
        return len(self.itos)

    def encode(self, tokens):
        return [self.stoi.get(token, UNK) for token in tokens]

    def decode(self, ids):
        tokens = []
        for idx in ids:
            if idx == EOS:
                break
            if idx not in (PAD, BOS):
                tokens.append(self.itos[idx])
        return tokens


class TranslationDataset(Dataset):
    def __init__(self, pairs, source_vocab, target_vocab):
        self.pairs = list(pairs)
        self.encoded = [
            (
                torch.tensor(source_vocab.encode(src.split()) + [EOS]),
                torch.tensor([BOS] + target_vocab.encode(tgt.split()) + [EOS]),
            )
            for src, tgt in self.pairs
        ]

    def __len__(self):
        return len(self.encoded)

    def __getitem__(self, index):
        return self.encoded[index]

def collate_batch(batch):
    sources, targets = zip(*batch)
    lengths = torch.tensor([len(src) for src in sources], dtype=torch.long)
    return (
        pad_sequence(sources, batch_first=True, padding_value=PAD),
        lengths,  # pack_padded_sequence cần lengths trên CPU.
        pad_sequence(targets, batch_first=True, padding_value=PAD),
    )

def unknown_rate(pairs, column, vocab):
    tokens = [token for pair in pairs for token in pair[column].split()]
    return sum(token not in vocab.stoi for token in tokens) / max(len(tokens), 1)


class LengthBucketSampler(Sampler):
    """Xáo trộn theo epoch, gom độ dài trong các pool 50 batch."""
    def __init__(self, dataset, batch_size, seed):
        self.lengths = [max(len(src), len(tgt)) for src, tgt in dataset.encoded]
        self.batch_size = batch_size
        self.seed = seed
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __len__(self):
        return math.ceil(len(self.lengths) / self.batch_size)

    def __iter__(self):
        rng = random.Random(self.seed + self.epoch)
        indices = list(range(len(self.lengths)))
        rng.shuffle(indices)
        batches = []
        pool_size = self.batch_size * 50
        for start in range(0, len(indices), pool_size):
            pool = sorted(indices[start:start + pool_size], key=self.lengths.__getitem__)
            batches.extend(pool[i:i + self.batch_size] for i in range(0, len(pool), self.batch_size))
        rng.shuffle(batches)
        yield from batches


In [ ]:
src_vocab = Vocabulary.build(
    (src.split() for src, _ in train_pairs), CFG["src_vocab_size"], CFG["min_freq"],
)
tgt_vocab = Vocabulary.build(
    (tgt.split() for _, tgt in train_pairs), CFG["tgt_vocab_size"], CFG["min_freq"],
)

train_dataset = TranslationDataset(train_pairs, src_vocab, tgt_vocab)
validation_dataset = TranslationDataset(validation_pairs, src_vocab, tgt_vocab)
test_dataset = TranslationDataset(test_pairs, src_vocab, tgt_vocab)
train_sampler = LengthBucketSampler(train_dataset, CFG["batch_size"], CFG["seed"])
loader_options = {
    "collate_fn": collate_batch,
    "num_workers": CFG["num_workers"],
    "pin_memory": DEVICE.type == "cuda",
    "persistent_workers": CFG["num_workers"] > 0,
}
train_loader = DataLoader(train_dataset, batch_sampler=train_sampler, **loader_options)
validation_loader = DataLoader(validation_dataset, batch_size=CFG["batch_size"], **loader_options)
test_loader = DataLoader(test_dataset, batch_size=CFG["batch_size"], **loader_options)

print(f"Vocabulary: EN={len(src_vocab):,} | VI={len(tgt_vocab):,}")
print(f"Train: {len(train_loader):,} batch/epoch")
print(f"Test UNK: EN={unknown_rate(test_pairs, 0, src_vocab):.2%}, VI={unknown_rate(test_pairs, 1, tgt_vocab):.2%}")
sample_src, sample_lengths, sample_tgt = collate_batch([train_dataset[i] for i in range(3)])


## 4. Encoder–Decoder GRU có Attention

**Ví dụ NLP:** encoder đọc token tiếng Anh → embedding → GRU hai chiều → memory và hidden khởi tạo decoder. Decoder đọc BOS/token Việt trước đó → GRU một chiều → Attention tới memory → logits từ vựng Việt.

Train dùng teacher forcing 100%, xử lý toàn chuỗi target đã dịch phải trong một lần gọi GRU. Attention và tầng dự đoán cũng tính theo batch. Khi dịch tự sinh, decoder sinh từng token, dừng tại EOS.

GRU chạy FP32 trên GPU để giữ input/hidden cùng dtype; các phép chiếu, attention và output dùng AMP. Loss bỏ PAD, attention mask PAD nguồn.

| Tensor | Shape |
|---|---|
| Source IDs | `(B, S)` |
| Encoder memory | `(B, S, H)` |
| Decoder states | `(B, T, H)` |
| Attention | `(B, T, S)` |
| Logits | `(B, T, V_vi)` |

API: [GRU](https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html), [AMP](https://docs.pytorch.org/docs/stable/notes/amp_examples.html), [DataLoader](https://docs.pytorch.org/tutorials/intermediate/intermediate_data_loading_tutorial.html).


In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD)
        self.dropout = nn.Dropout(dropout)
        self.gru = nn.GRU(
            embedding_dim, hidden_dim, batch_first=True, bidirectional=True
        )
        self.memory_projection = nn.Linear(2 * hidden_dim, hidden_dim)
        self.hidden_bridge = nn.Linear(2 * hidden_dim, hidden_dim)

    def forward(self, src, lengths):
        with torch.autocast(device_type=src.device.type, enabled=False):
            embedded = self.dropout(self.embedding(src))
            packed = pack_padded_sequence(
                embedded, lengths, batch_first=True, enforce_sorted=False
            )
            packed_output, hidden = self.gru(packed)
            output, _ = pad_packed_sequence(
                packed_output, batch_first=True, total_length=src.size(1)
            )
        memory = torch.tanh(self.memory_projection(output))
        last_hidden = torch.cat((hidden[-2], hidden[-1]), dim=-1)
        initial_decoder_hidden = torch.tanh(self.hidden_bridge(last_hidden)).unsqueeze(0)
        return memory, initial_decoder_hidden, src.ne(PAD)

class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD)
        self.dropout = nn.Dropout(dropout)
        self.gru = nn.GRU(embedding_dim, hidden_dim, batch_first=True)
        self.query_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.fusion = nn.Linear(2 * hidden_dim, hidden_dim)
        self.output_projection = nn.Linear(hidden_dim, vocab_size)
        self.scale = math.sqrt(hidden_dim)

    def forward(self, input_ids, hidden, memory, source_mask):
        with torch.autocast(device_type=input_ids.device.type, enabled=False):
            embedded = self.dropout(self.embedding(input_ids))
            states, hidden = self.gru(embedded, hidden.float())
        scores = torch.bmm(
            self.query_projection(states), memory.transpose(1, 2)
        ) / self.scale
        scores = scores.masked_fill(~source_mask.unsqueeze(1), float("-inf"))
        attention = scores.float().softmax(dim=-1).to(memory.dtype)
        context = torch.bmm(attention, memory)
        fused = torch.tanh(self.fusion(torch.cat((states, context), dim=-1)))
        logits = self.output_projection(self.dropout(fused))
        return logits, hidden, attention

class Seq2Seq(nn.Module):
    def __init__(self, src_size, tgt_size, embedding_dim, hidden_dim, dropout):
        super().__init__()
        self.encoder = Encoder(src_size, embedding_dim, hidden_dim, dropout)
        self.decoder = Decoder(tgt_size, embedding_dim, hidden_dim, dropout)

    def forward(self, src, lengths, decoder_inputs):
        memory, hidden, mask = self.encoder(src, lengths)
        logits, _, attention = self.decoder(decoder_inputs, hidden, memory, mask)
        return logits, attention

def make_model(config, source_vocab, target_vocab):
    return Seq2Seq(
        len(source_vocab), len(target_vocab),
        config["embedding_dim"], config["hidden_dim"], config["dropout"],
    )


In [ ]:
set_seed(CFG["seed"])
model = make_model(CFG, src_vocab, tgt_vocab).to(DEVICE)
criterion = nn.CrossEntropyLoss(ignore_index=PAD, reduction="sum")
model.eval()
with torch.no_grad(), torch.autocast(DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
    src = sample_src.to(DEVICE)
    memory, initial_hidden, _ = model.encoder(src, sample_lengths)
    embeddings = model.encoder.embedding(src)
    sample_logits, sample_attention = model(src, sample_lengths, sample_tgt[:, :-1].to(DEVICE))

print("Model device:", next(model.parameters()).device)
print(f"Tham số: {sum(p.numel() for p in model.parameters()):,}")
print("EN:", train_pairs[0][0])
print("VI:", train_pairs[0][1])
print("Embedding:", tuple(embeddings.shape), "| Encoder memory:", tuple(memory.shape))
print("Decoder initial hidden:", tuple(initial_hidden.shape))
print("Logits:", tuple(sample_logits.shape), "| Attention:", tuple(sample_attention.shape))
print("8 chiều đầu hidden:", initial_hidden[0, 0, :8].float().cpu().tolist())
# Giải phóng tensor minh họa trước khi train.
del src, memory, initial_hidden, embeddings, sample_logits, sample_attention


## 5. Train, early stopping và checkpoint

Adam + AMP/GradScaler + gradient clipping. Loss trung bình theo số token đích khác PAD, gồm EOS. Train dùng teacher forcing; token accuracy không tương đương chất lượng bản dịch tự sinh.

Thống kê cộng trên thiết bị, chỉ cập nhật loss hiển thị mỗi 50 batch. Mỗi epoch lưu `last_model.pt`; validation loss thấp nhất lưu `best_model.pt`. File checkpoint được ghi tạm rồi thay thế. Để resume, đặt `RESUME_PATH` tới **last_model.pt** ở cell cấu hình và chỉnh `epochs` thành tổng số epoch muốn chạy. Cấu hình model/dữ liệu được lấy lại từ checkpoint.


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=CFG["learning_rate"])
scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)


def run_epoch(loader, optimizer=None, description=""):
    training = optimizer is not None
    model.train(training)
    totals = torch.zeros(3, dtype=torch.float64, device=DEVICE)
    progress = tqdm(loader, desc=description, leave=False, mininterval=1.0)
    for step, (src, lengths, tgt) in enumerate(progress, 1):
        src = src.to(DEVICE, non_blocking=True)
        tgt = tgt.to(DEVICE, non_blocking=True)
        labels = tgt[:, 1:]
        valid = labels.ne(PAD)
        token_count = valid.sum()
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                logits, _ = model(src, lengths, tgt[:, :-1])
                loss_sum = criterion(logits.reshape(-1, len(tgt_vocab)), labels.reshape(-1))
                loss = loss_sum / token_count
            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), CFG["gradient_clip"])
                scaler.step(optimizer)
                scaler.update()
        correct = ((logits.detach().argmax(-1) == labels) & valid).sum()
        totals += torch.stack((loss_sum.detach(), correct, token_count)).to(torch.float64)
        if step % 50 == 0:
            progress.set_postfix(loss=f"{(totals[0] / totals[2]).item():.3f}")
        del logits, loss_sum, loss
    loss_total, correct_total, tokens_total = totals.tolist()
    return {"loss": loss_total / tokens_total,
            "token_accuracy": correct_total / tokens_total, "tokens": int(tokens_total)}


def save_checkpoint(payload, path):
    temporary = path.with_suffix(".pt.tmp")
    torch.save(payload, temporary)
    temporary.replace(path)


BEST_PATH = RUN_DIR / "best_model.pt"
LAST_PATH = RUN_DIR / "last_model.pt"
history = []
best_val_loss, best_epoch, stale_epochs, start_epoch = float("inf"), 0, 0, 1
best_snapshot = None
if resume is not None:
    model.load_state_dict(resume["model_state"])
    optimizer.load_state_dict(resume["optimizer_state"])
    scaler.load_state_dict(resume["scaler_state"])
    history = list(resume["history"])
    best_snapshot = resume["best_checkpoint"]
    best_val_loss = best_snapshot["validation_loss"]
    best_epoch = best_snapshot["epoch"]
    stale_epochs = resume["stale_epochs"]
    start_epoch = resume["epoch"] + 1
    save_checkpoint(best_snapshot, BEST_PATH)
    print(f"Resume từ epoch {resume['epoch']}; chạy tiếp từ epoch {start_epoch}.")

write_json(RUN_DIR / "config.json", CFG)
write_json(RUN_DIR / "data_summary.json", split_summary)
write_json(RUN_DIR / "vocab.json", {"src_itos": src_vocab.itos, "tgt_itos": tgt_vocab.itos})

for epoch in range(start_epoch, CFG["epochs"] + 1):
    # Seed theo epoch giúp resume giữ thứ tự batch và dropout của epoch đó.
    set_seed(CFG["seed"] + epoch)
    train_sampler.set_epoch(epoch)
    started = time.perf_counter()
    train_metrics = run_epoch(train_loader, optimizer, f"Train {epoch}/{CFG['epochs']}")
    val_metrics = run_epoch(validation_loader, description="Validation")
    elapsed = time.perf_counter() - started
    state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    improved = val_metrics["loss"] < best_val_loss
    if improved:
        best_val_loss, best_epoch, stale_epochs = val_metrics["loss"], epoch, 0
        best_snapshot = {
            "model_state": state, "config": dict(CFG),
            "src_itos": src_vocab.itos, "tgt_itos": tgt_vocab.itos,
            "epoch": epoch, "validation_loss": best_val_loss,
            "data_summary": split_summary, "torch_version": str(torch.__version__),
        }
        save_checkpoint(best_snapshot, BEST_PATH)
    else:
        stale_epochs += 1
    history.append({
        "epoch": epoch, "train_loss": train_metrics["loss"],
        "validation_loss": val_metrics["loss"],
        "train_token_accuracy": train_metrics["token_accuracy"],
        "validation_token_accuracy": val_metrics["token_accuracy"], "seconds": elapsed,
    })
    save_checkpoint({
        "model_state": state, "optimizer_state": optimizer.state_dict(),
        "scaler_state": scaler.state_dict(), "config": dict(CFG),
        "epoch": epoch, "history": history, "stale_epochs": stale_epochs,
        "best_checkpoint": best_snapshot,
    }, LAST_PATH)
    write_json(RUN_DIR / "history.json", history)
    print(f"Epoch {epoch:02d} | train={train_metrics['loss']:.4f} | "
          f"val={val_metrics['loss']:.4f} | val acc={val_metrics['token_accuracy']:.2%} | "
          f"{elapsed:.1f}s" + (" | lưu best + last" if improved else " | lưu last"))
    if stale_epochs >= CFG["patience"]:
        print("Early stopping.")
        break

print(f"Best epoch: {best_epoch} | val loss: {best_val_loss:.4f}")
print("Model để dịch:", BEST_PATH)
print("Checkpoint để resume:", LAST_PATH)


In [ ]:
best_checkpoint = torch.load(BEST_PATH, map_location="cpu", weights_only=True)
model.load_state_dict(best_checkpoint["model_state"])
model.to(DEVICE).eval()
print("Đã lấy model tại epoch:", best_checkpoint["epoch"])

epochs = [row["epoch"] for row in history]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(epochs, [r["train_loss"] for r in history], "o-", label="Train")
axes[0].plot(epochs, [r["validation_loss"] for r in history], "o-", label="Validation")
axes[0].set(xlabel="Epoch", ylabel="Cross-entropy / token", title="Loss (teacher forcing)")
axes[1].plot(
    epochs, [r["validation_token_accuracy"] for r in history], "o-", color="tab:green"
)
axes[1].set(xlabel="Epoch", ylabel="Accuracy", title="Validation token accuracy")
axes[0].legend()
plt.tight_layout()
plt.show()


## 6. Greedy decoding và đánh giá test

Decoder bắt đầu bằng BOS, dùng token tự sinh làm input tiếp theo và dừng tại EOS hoặc giới hạn sinh. PAD/BOS bị chặn trong đầu ra. Reference tiếng Việt chỉ dùng tính metric, không đưa vào decoder khi dịch.

Tính loss/accuracy theo teacher forcing và BLEU/chrF trên bản dịch tự sinh. Kết quả áp dụng cho test sau lọc độ dài. Câu nhập để dịch dài hơn giới hạn source sẽ được cắt token theo `max_src_tokens`.


In [ ]:
@torch.inference_mode()
def greedy_decode(
    texts, current_model, source_vocab, target_vocab,
    max_source_tokens, max_new_tokens, return_attention=False,
):
    if not texts:
        return ([], ([], []), None) if return_attention else []
    current_model.eval()
    device = next(current_model.parameters()).device
    token_lists = [tokenize(text)[:max_source_tokens] for text in texts]
    sources = [torch.tensor(source_vocab.encode(tokens) + [EOS]) for tokens in token_lists]
    lengths = torch.tensor([len(src) for src in sources])
    src = pad_sequence(sources, batch_first=True, padding_value=PAD).to(device)
    next_input = torch.full((len(texts), 1), BOS, dtype=torch.long, device=device)
    finished = torch.zeros(len(texts), dtype=torch.bool, device=device)
    generated, attention_steps = [], []
    with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == "cuda"):
        memory, hidden, source_mask = current_model.encoder(src, lengths)
        for _ in range(max_new_tokens):
            logits, hidden, attention = current_model.decoder(next_input, hidden, memory, source_mask)
            scores = logits[:, -1]
            scores[:, [PAD, BOS]] = float("-inf")
            next_ids = scores.argmax(-1).masked_fill(finished, PAD)
            generated.append(next_ids)
            if return_attention:
                attention_steps.append(attention[:, 0])
            finished |= next_ids.eq(EOS)
            next_input = next_ids.unsqueeze(1)
            if bool(finished.all()):
                break
    generated = torch.stack(generated, dim=1).cpu().tolist()
    predictions = [tokens_to_text(target_vocab.decode(ids)) for ids in generated]
    if return_attention:
        matrix = torch.stack(attention_steps, dim=1).float().cpu().numpy()
        source_labels = [tokens + ["<eos>"] for tokens in token_lists]
        return predictions, (generated, source_labels), matrix
    return predictions


In [ ]:
test_metrics = run_epoch(test_loader, description="Test loss")
print(
    f"Test loss (teacher forcing)={test_metrics['loss']:.4f} | "
    f"perplexity={math.exp(min(test_metrics['loss'], 50)):.2f} | "
    f"token accuracy={test_metrics['token_accuracy']:.2%}"
)

hypotheses = []
for start in tqdm(
    range(0, len(test_pairs), CFG["batch_size"]), desc="Greedy dịch test"
):
    batch = test_pairs[start:start + CFG["batch_size"]]
    hypotheses.extend(greedy_decode(
        [src for src, _ in batch], model, src_vocab, tgt_vocab,
        CFG["max_src_tokens"], CFG["max_tgt_tokens"] + 1,
    ))
references = [tokens_to_text(tgt.split()) for _, tgt in test_pairs]
bleu_metric, chrf_metric = BLEU(tokenize="13a"), CHRF()
bleu_score = bleu_metric.corpus_score(hypotheses, [references])
chrf_score = chrf_metric.corpus_score(hypotheses, [references])
print(f"Test: {len(test_pairs):,} cặp sau lọc/lấy mẫu.")
print("BLEU:", bleu_score)
print("chrF:", chrf_score)
print("BLEU signature:", bleu_metric.get_signature())
print("chrF signature:", chrf_metric.get_signature())

evaluation = {
    "checkpoint_epoch": best_checkpoint["epoch"],
    "test_pairs": len(test_pairs),
    "teacher_forced_loss": test_metrics["loss"],
    "teacher_forced_token_accuracy": test_metrics["token_accuracy"],
    "bleu": bleu_score.score,
    "chrf": chrf_score.score,
    "bleu_signature": str(bleu_metric.get_signature()),
    "chrf_signature": str(chrf_metric.get_signature()),
}
write_json(RUN_DIR / "metrics.json", evaluation)
with (RUN_DIR / "test_predictions.tsv").open(
    "w", encoding="utf-8", newline=""
) as handle:
    writer = csv.writer(handle, delimiter="\t", quoting=csv.QUOTE_MINIMAL)
    writer.writerow(["source_en", "reference_vi", "prediction_vi"])
    for (src, _), reference, hypothesis in zip(test_pairs, references, hypotheses):
        writer.writerow([src, reference, hypothesis])

for i in range(min(5, len(test_pairs))):
    print(f"\n[{i + 1}] EN: {tokens_to_text(test_pairs[i][0].split())}")
    print("REF:", references[i])
    print("PRED:", hypotheses[i])


## 7. Dịch câu của bạn và xem attention

Sửa `your_sentence`, rồi chạy lại cell. Chữ thường và dấu câu được xử lý bằng tokenizer đã dùng khi train. Các từ ngoài vocabulary thành UNK.

Attention là trọng số mô hình học được, không phải căn chỉnh ngôn ngữ được gán nhãn. Nếu bản dịch lặp từ, dừng sớm hoặc BLEU thấp sau cấu hình thử nhỏ, hãy xem loss và tăng lượng dữ liệu/số epoch trước khi đánh giá khả năng của kiến trúc.

In [ ]:
def translate(text, return_attention=False):
    result = greedy_decode(
        [text], model, src_vocab, tgt_vocab,
        CFG["max_src_tokens"], CFG["max_tgt_tokens"] + 1,
        return_attention=return_attention,
    )
    if return_attention:
        predictions, (generated, source_labels), attention = result
        target_labels = []
        for idx in generated[0]:
            target_labels.append(tgt_vocab.itos[idx])
            if idx == EOS:
                break
        matrix = attention[0, :len(target_labels), :len(source_labels[0])]
        return predictions[0], source_labels[0], target_labels, matrix
    return result[0]

your_sentence = "I am very happy to be here today."
prediction, source_labels, target_labels, attention_matrix = translate(
    your_sentence, return_attention=True
)
print("EN:", your_sentence)
print("VI:", prediction)
print(
    "Token EN ngoài vocabulary:",
    [token for token in tokenize(your_sentence) if token not in src_vocab.stoi],
)

fig, ax = plt.subplots(
    figsize=(max(7, len(source_labels) * 0.45), max(3, len(target_labels) * 0.25))
)
heatmap = ax.imshow(attention_matrix, aspect="auto", cmap="Blues")
ax.set_xticks(range(len(source_labels)), source_labels, rotation=60, ha="right")
ax.set_yticks(range(len(target_labels)), target_labels)
ax.set(xlabel="Token tiếng Anh", ylabel="Token decoder sinh", title="Attention khi dịch")
fig.colorbar(heatmap, ax=ax, label="Trọng số")
plt.tight_layout()
plt.show()


## 8. Tải checkpoint để dịch

Sau Run All, cell dưới minh họa tải lại `best_model.pt`. Với phiên Kaggle mới: chạy cell import, các định nghĩa tokenizer/vocabulary/model/greedy decoding; sửa `CHECKPOINT_PATH` thành đường dẫn checkpoint đã giữ rồi chạy cell này. Hàm `load_translator` nhận `device="cuda"` mặc định.


In [ ]:
def load_translator(checkpoint_path, device="cuda"):
    saved = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    source_vocab = Vocabulary(saved["src_itos"])
    target_vocab = Vocabulary(saved["tgt_itos"])
    restored = make_model(saved["config"], source_vocab, target_vocab).to(device)
    restored.load_state_dict(saved["model_state"])
    restored.eval()
    return restored, source_vocab, target_vocab, saved["config"]


CHECKPOINT_PATH = BEST_PATH
# Phiên mới: CHECKPOINT_PATH = Path("/kaggle/working/.../best_model.pt")
saved_model, saved_src_vocab, saved_tgt_vocab, saved_config = load_translator(CHECKPOINT_PATH)


def translate_saved(text):
    return greedy_decode(
        [text], saved_model, saved_src_vocab, saved_tgt_vocab,
        saved_config["max_src_tokens"], saved_config["max_tgt_tokens"] + 1,
    )[0]


print("Checkpoint:", CHECKPOINT_PATH)
print("Dịch từ model đã tải:", translate_saved("I am very happy to be here today."))


## 9. Chỉnh cấu hình và tiếp tục train

- **Train mới:** `RESUME_PATH = None`, sửa `CFG`, rồi Run All.
- **Resume:** đặt `RESUME_PATH` tới `last_model.pt`; `CFG["epochs"]` là tổng số epoch, phải lớn hơn epoch đã lưu. Giữ cùng bộ TSV; checkpoint khôi phục config model/dữ liệu, optimizer, scaler, best model và lịch sử.
- **Thiếu VRAM:** giảm `batch_size` từ 64 xuống 32 trước khi train mới.
- **Thử nhanh:** giới hạn số mẫu và đặt 3 epoch. Kết quả thử nhỏ phục vụ kiểm tra pipeline, không phản ánh chất lượng train đầy đủ.

Tệp kết quả: `best_model.pt`, `last_model.pt`, `config.json`, `vocab.json`, `data_summary.json`, `history.json`, `metrics.json`, `test_predictions.tsv`. Checkpoint ghi sau khi hoàn tất epoch; epoch đang chạy dở sẽ chạy lại khi resume. Lưu các tệp cần giữ trước khi kết thúc phiên Kaggle.

Baseline này dùng tokenizer theo token/âm tiết. Khi so sánh subword hoặc Transformer, giữ cùng split và cách tính metric.
